# Space-map Quickstart

Register a small stack of serial sections with the stable `space_map.api` facade. Uses the bundled toy dataset (`examples/toy_data.csv.gz`, 5 layers, ~10k cells) and runs in well under a minute on CPU.

In [ ]:
import pandas as pd
from space_map.api import register, RegistrationConfig

df = pd.read_csv("toy_data.csv.gz")
xys = [g[["x", "y"]].to_numpy(float) for _, g in df.groupby("layer", sort=True)]
print("layers:", [a.shape for a in xys])

## Run registration

`register()` runs the two-stage pipeline (affine + non-rigid) and returns aligned coordinates plus a QC report and a reproducibility manifest.

In [ ]:
result = register(xys, RegistrationConfig(workdir="quickstart-out", seed=0, device="cpu"))
result.save("quickstart-out")
print("aligned:", [a.shape for a in result.aligned])
print("timings:", result.manifest["timings"])

## Inspect quality control

The QC report gives adjacent-layer bidirectional Chamfer distance (a self-consistency measure, not an independent accuracy metric), finiteness, and dropped-cell accounting.

In [ ]:
import json
print(json.dumps(result.qc["internal_consistency"]["pairs"], indent=2, default=float))
print("all finite:", result.qc["finiteness"]["all_finite"])
print("warnings:", result.warnings)

## Outputs

`quickstart-out/` now contains `aligned/<layer>.npy` and JSON files for `qc`, `manifest`, `transforms`, and `warnings`.